#### **01 — Engenharia de Atributos**

**Objetivo:** criar os atributos comportamentais e agregados que serão usados no binning e na modelagem.

**Entrada:** `data/01_prata/*_tratado.parquet`  
**Saída:** `data/01_prata/*_fe.parquet`

As features são criadas **depois do tratamento** e **sem usar o target**.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

#### **1. Carga das bases tratadas**

In [2]:
treino = pd.read_parquet("../data/01_prata/treino_tratado.parquet")
validacao = pd.read_parquet("../data/01_prata/validacao_tratado.parquet")
teste = pd.read_parquet("../data/01_prata/teste_tratado.parquet")

print("treino:", treino.shape)
print("validacao:", validacao.shape)
print("teste:", teste.shape)

treino: (18000, 26)
validacao: (6000, 26)
teste: (6000, 26)


#### **2. Criação dos atributos**

Os agregados de `BILL_AMT` e `PAY_AMT` são calculados sobre os valores já tratados.

In [3]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

def criar_atributos(base):
    return base.assign(
        SUM_BILL=lambda x: x[bill_cols].sum(axis=1),
        AVG_BILL=lambda x: x[bill_cols].mean(axis=1),
        MAX_BILL=lambda x: x[bill_cols].max(axis=1),

        SUM_PAY=lambda x: x[pay_amt_cols].sum(axis=1),
        AVG_PAY=lambda x: x[pay_amt_cols].mean(axis=1),
        MAX_PAY=lambda x: x[pay_amt_cols].max(axis=1),

        UTILIZACAO=lambda x: (x["BILL_AMT1"] / x["LIMIT_BAL"].replace(0, np.nan)).clip(0, 2),
        UTIL_MEDIA=lambda x: (x[bill_cols].mean(axis=1) / x["LIMIT_BAL"].replace(0, np.nan)).clip(0, 2),
        PCT_PAGAMENTO=lambda x: (x["PAY_AMT1"] / x["BILL_AMT1"].replace(0, np.nan)).clip(0, 2),
        FREQ_ATRASO=lambda x: (x[pay_cols] > 0).sum(axis=1),
        MAX_ATRASO=lambda x: x[pay_cols].max(axis=1),
        LIMIT_PER_CAPITA=lambda x: x["LIMIT_BAL"] / x["AGE"].replace(0, np.nan),
    )

#### **3. Aplicação e gravação**

In [4]:
treino_fe = criar_atributos(treino)
validacao_fe = criar_atributos(validacao)
teste_fe = criar_atributos(teste)

treino_fe.to_parquet("../data/01_prata/treino_fe.parquet")
validacao_fe.to_parquet("../data/01_prata/validacao_fe.parquet")
teste_fe.to_parquet("../data/01_prata/teste_fe.parquet")

print("treino_fe:", treino_fe.shape)
print("validacao_fe:", validacao_fe.shape)
print("teste_fe:", teste_fe.shape)

print("\nAtributos criados:")
print([
    "SUM_BILL", "AVG_BILL", "MAX_BILL",
    "SUM_PAY", "AVG_PAY", "MAX_PAY",
    "UTILIZACAO", "UTIL_MEDIA", "PCT_PAGAMENTO",
    "FREQ_ATRASO", "MAX_ATRASO", "LIMIT_PER_CAPITA",
])

treino_fe: (18000, 38)
validacao_fe: (6000, 38)
teste_fe: (6000, 38)

Atributos criados:
['SUM_BILL', 'AVG_BILL', 'MAX_BILL', 'SUM_PAY', 'AVG_PAY', 'MAX_PAY', 'UTILIZACAO', 'UTIL_MEDIA', 'PCT_PAGAMENTO', 'FREQ_ATRASO', 'MAX_ATRASO', 'LIMIT_PER_CAPITA']


In [5]:
treino_fe.head()

,ID,SAFRA,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default,SUM_BILL,AVG_BILL,MAX_BILL,SUM_PAY,AVG_PAY,MAX_PAY,UTILIZACAO,UTIL_MEDIA,PCT_PAGAMENTO,FREQ_ATRASO,MAX_ATRASO,LIMIT_PER_CAPITA
0,1,200510,20000,2,2,1,24,2.0,2.0,-1.0,-1.0,-2.0,-2.0,3913,3102,689,0,0,0,0.0,689.0,0.0,0.0,0.0,0.0,1,7704,1284.000000,3913,689.0,114.833333,689.0,0.195650,0.064200,0.000000,2,2.0,833.333333
1,2,200510,120000,2,2,2,26,-1.0,2.0,0.0,0.0,0.0,2.0,2682,1725,2682,3272,3455,3261,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1,17077,2846.166667,3455,5000.0,833.333333,2000.0,0.022350,0.023718,0.000000,2,2.0,4615.384615
2,3,200510,90000,2,2,2,34,0.0,0.0,0.0,0.0,0.0,0.0,29239,14027,13559,14331,14948,15549,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0,101653,16942.166667,29239,11018.0,1836.333333,5000.0,0.324878,0.188246,0.051917,0,0.0,2647.058824
3,4,200510,50000,2,2,1,37,0.0,0.0,0.0,0.0,0.0,0.0,46990,48233,49291,28314,28959,29547,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0,231334,38555.666667,49291,8388.0,1398.000000,2019.0,0.939800,0.771113,0.042562,0,0.0,1351.351351
4,5,200510,50000,1,2,1,57,-1.0,0.0,-1.0,0.0,0.0,0.0,8617,5670,35835,20940,19146,19131,2000.0,10048.2,9154.3,9000.0,689.0,679.0,0,109339,18223.166667,35835,31570.5,5261.750000,10048.2,0.172340,0.364463,0.232099,0,0.0,877.192982
